In [ ]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'  # Must be set BEFORE importing JAX

import jax
jax.config.update('jax_enable_x64', True)  # Enable 64-bit floats

import jax.numpy as jnp
from difflow import CSTR, make_stream
from difflow.units.cstr import CSTRParams

# Define rate function: r = k * C_A where k = A * exp(-Ea/RT)
def rate_fn(C, T, params):
    k = params['A'] * jnp.exp(-params['Ea'] / (8.314 * T))
    r_A = k * C["A"]
    return jnp.array([r_A])  # Returns array of reaction rates

# Stoichiometry: A -> B (one reaction)
stoich = jnp.array([[-1], [1]])  # Shape: (n_species, n_reactions)
species_order = ["A", "B"]

# Kinetic parameters
rate_params = {'A': 1000.0, 'Ea': 5000.0 * 8.314}  # Ea in J/mol

# Create inlet stream
inlet_stream = make_stream({"A": 1.0, "B": 0.0}, T=350.0, P=101325.0)

# Create base params (V will be replaced)
base_params = CSTRParams(
    V=1.0,  # Placeholder
    rate_fn=rate_fn,
    stoich=stoich,
    rate_params=rate_params,
    species_order=species_order,
)

def conversion(volume):
    params = base_params.update(V=volume)
    reactor = CSTR(params)
    outlet, info = reactor(inlet_stream)
    return info["conversion"]["A"]

# Exact gradient via automatic differentiation
dX_dV = jax.grad(conversion)(2.0)  # dConversion/dVolume
dX_dV

In [ ]:
def experiment(params):
    """Returns conversion as function of (V, A) parameters."""
    cstr_params = base_params.update(**params)
    reactor = CSTR(cstr_params)
    outlet, info = reactor(inlet_stream)
    return info["conversion"]["A"]

# Parameters to differentiate with respect to
params = {'V': 2.0, 'rate_params': {'A': 1000.0, 'Ea': 5000.0 * 8.314}}

# Compute gradient with respect to all parameters
sensitivities = jax.grad(experiment)(params)
print(f"dX/dV = {sensitivities['V']:.6f}")
print(f"dX/dA = {sensitivities['rate_params']['A']:.6f}")